RNN - Erro dos pesos computados e usado somente durante a iteração

In [1]:
import numpy as np
from numpy import linalg as LA
import pandas as pd
import ipynbname
import optuna
from optuna.samplers import RandomSampler
from optuna.samplers import TPESampler
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_pareto_front
from optuna.importance import get_param_importances
from optuna.exceptions import TrialPruned
import matplotlib as mpl
from sklearn.preprocessing import MinMaxScaler
from Functions.RLS import *
from Functions.Utils_RTLO import *
from Functions.Graphs import *
from sklearn.metrics import root_mean_squared_error as RMSE
from sklearn.metrics import mean_absolute_percentage_error as MAPE
from sklearn.metrics import mean_squared_error as MSE
FileName = ipynbname.name()

path = r'Datasets\DEVRT\NISSAN LEAF\20230421_NISSAN_DONOSTIA_ULIA_056.csv'
df = pd.read_csv(path)
df.fillna(0, inplace=True)
df.head()

pwr = (df['Motor Pwr(w)'].values)
spd = (df['speed'].values)
rpm = (df['rpm'].values)
trq = (df['Torque Nm'].values)
data = pd.DataFrame({'speed': spd, 'rpm': rpm, 'torque': trq, 'power': pwr,})
data = data.values
scaler = MinMaxScaler()
data_norm = scaler.fit_transform(data)

sig = data.T[-1]

def SelSampler(mode='auto'):
    '''mode: auto, random,  tpe'''
    if mode == 'auto':
        sampler = None
    elif mode == 'tpe':
        sampler = optuna.samplers.TPESampler(multivariate=True, constant_linker=True,group=True,n_startup_trials=2000)
    elif mode == 'random':
        sampler=RandomSampler()
    return sampler


In [2]:
class RTLO:
    def __init__(self, nI,nR,nO,ηS=[0.1,0.1,0.1], τ=10,mode='past'):
        np.random.seed(42)
        self.k = 1
        self.j = nI-1
        self.t = np.array([])
        self.start = 0
        self.ref = None
        self.act = 'tanh'
        self.flw = mode
        self.n = -1
        self.nI, self.nR, self.nO = nI, nR, nO

        self.ηS = np.array(ηS)
        self.τ = τ
        self.ρ = 0.003

        self.x = np.zeros(nI)
        self.hP, self.hU, self.hL = [0.1*np.ones(nR) for i in range(3)]
        self.hP2, self.hU2, self.hL2 = [0.1*np.ones(nR) for i in range(3)]
        self.uP, self.uP2 =  [np.zeros(nR) for i in range(2)]
        self.pS = np.zeros((self.nR, self.nR))
        self.qS = np.zeros((self.nR, self.nI))
        
        self.wI = XavierUniform([nR, nI],sd=42)
        self.wR = XavierUniform([nR, nR],sd=41)
        self.wO = XavierUniform([nO, nR],sd=40)
        self.BS = XavierUniform([nR, nO],sd=39)
    
        self.yP, self.yR, self.yL, self.yU = [np.array([]) for i in range(4)]
        self.yP_hist = np.zeros(self.nI)
        self.εY, self.εM, self.εR, self.εE, self.eP, self.eR, self.ΣW = [0 for i in range(7)]
        self.εM_hist,self.εR_hist, self.eR_hist, self.eP_hist = [np.array([]) for i in range(4)]
        self.wR_hist = []
        self.wI_hist = []
        self.wO_hist = []

        self.rR = 1e-9
        self.rP = 1e-10
        self.rL = 1e-11
        self.rU = 1e-12
        self.rRsum = 0
        self.rulR, self.rulP, self.rulL, self.rulU = [np.array([]) for i in range(4)]


    def PredSingle(self,x):

        u = np.dot(self.wR, self.hP) + np.dot(self.wI, x)
        h = self.hP + (-self.hP + Activation(u,self.act))/self.τ
        y = np.dot(self.wO, h)

        return y

    def fit(self,xP,yR):
        if self.flw != 'past': self.n = 0
        
        η1,η2,η3 = self.ηS      
        uP = self.wR @ self.hP + self.wI @ xP
        hP = self.hP*(1-1/self.τ) + Activation(uP,self.act)/self.τ
        yP = self.wO @ hP
        eS = yR-yP

        self.pS = np.outer(dActivation(self.uP,self.act),self.hP)/self.τ + (1-1/self.τ)*self.pS
        self.qS = np.outer(dActivation(self.uP,self.act),self.x)/self.τ + (1-1/self.τ)*self.qS

        δOS = η1*np.outer(eS,hP)
        δRS = η2*np.outer((self.BS@eS),np.ones(self.nR))*self.pS
        δIS = η3*np.outer(np.dot(self.BS, eS),np.ones(self.nI))*self.qS

        self.wI = self.wI + δIS
        self.wR = self.wR + δRS
        self.wO = self.wO + δOS

        self.wR_hist.append(self.wR.flatten())
        self.wI_hist.append(self.wI.flatten())
        self.wO_hist.append(self.wO.flatten())

        self.hP = hP
        self.hP2 = hP
        self.hL = hP
        self.hU = hP
        self.uP = uP
        self.x = xP

        if self.k == 1:
            self.yP = yP.reshape(-1,self.nO)
            self.yR = yR.reshape(-1,self.nO)
        else:
            self.yP = np.vstack((self.yP,yP))
            self.yR = np.vstack((self.yR,yR))
        self.t = np.append(self.t,self.k)
        self.k = self.k+1
        
        #self.ηS = self.ηS/(1 + self.decay*self.k
    
    def Predict(self, x):
        if self.flw != 'past': self.n = 0
        xP = x.copy()
       
        
        hP = self.hP2.copy()
        #print('P S',hP[:5])
        uP = (self.wR @ hP) + (self.wI @ xP)
        hP = hP*(1-1/self.τ) + Activation(uP,self.act)/self.τ
        yP = (self.wO @ hP)[self.n]
        self.hP2 = hP
        return yP
    
    def PredictIntr(self,xP,xL,xU,ep,show=False):
        if self.flw != 'past': self.n = 0
        wR,wI,wO = self.wR,self.wI,self.wO
        hP,hU,hL = self.hP2.copy(),self.hU2.copy(),self.hL2.copy()

        if show:
            print('hP antes:', hP[-5:])

        wRU, wRL = np.maximum((1+ep)*wR, wR/(1+ep)), np.minimum((1+ep)*wR, wR/(1+ep))
        wIU, wIL = np.maximum((1+ep)*wI, wI/(1+ep)), np.minimum((1+ep)*wI, wI/(1+ep))
        wOU, wOL = np.maximum((1+ep)*wO, wO/(1+ep)), np.minimum((1+ep)*wO, wO/(1+ep))
        
        uP = ( wR @ hP) + ( wI @ xP)
        uL = (wRL @ hL) + (wIL @ xL)
        uU = (wRU @ hU) + (wIU @ xU)

        
        uU, uL = np.maximum(uU,uL), np.minimum(uU,uL)
        
        hP = hP*(1-1/self.τ) + Activation(uP,self.act)/self.τ
        hL = hL*(1-1/self.τ) + Activation(uL,self.act)/self.τ
        hU = hU*(1-1/self.τ) + Activation(uU,self.act)/self.τ
        hU, hL = np.maximum(hU,hL), np.minimum(hU,hL)

        yP = ( wO @ hP)
        yL = (wOL @ hL)
        yU = (wOU @ hU)
        yU, yL = np.maximum(yU, yL), np.minimum(yU, yL)

        yP = yP[self.n]
        yL = yL[self.n]
        yU = yU[self.n]

        if show:
            print('uP:',uP[-5:])
            print('hP depois:',hP[-5:])
            print('xP antes:', xP[-3:],'yP:',yP)
            print('-----------------')

        

        self.hP2 = hP
        self.hL2 = hL
        self.hU2 = hU

        return np.array([yL,yP,yU])

    def Restore(self):
        self.hP2 = self.hP
        self.hL2 = self.hP
        self.hU2 = self.hP

    def ReturnParameters(self):

        return [self.wR,self.wI,self.wO,self.pS,self.qS,self.hP,self.hP2,self.hL2,self.hU2,self.x]
    
    def ReceiveParameters(self,vec):
        self.wR,self.wI,self.wO,self.pS,self.qS,self.hP,self.hP2,self.hL2,self.hU2,self.x = vec
        



 
        


#Optimize parameters for minimize error of degradation prediction

In [7]:
rates = [1/(10**i) for i in range(1,7)][::-1]
def objective(trial):

    nI = trial.suggest_int('nI', 2, 45) 
    nR = trial.suggest_int('nR', 2, 45) 
    nO = trial.suggest_int('nO', 5, 10) 
    N1 = trial.suggest_categorical('N1', rates[:]) 
    N2 = trial.suggest_categorical('N2', rates[:]) 
    N3 = trial.suggest_categorical('N3', rates) 
    τ = trial.suggest_int('τ', 1, 30)        
    X,Y = PrepareDataAhead(sig,n=nI,m=nO)
    rnn = RTLO(nI,nR,nO,[N1,N2,N3],τ)
    rnn.ref = len(sig)-nI

    for i,_ in enumerate(X):
        rnn.fit(X[i],Y[i])
            
    #return rnn.εY
    return MAPE(rnn.yR,rnn.yP)

#pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
pruner=optuna.pruners.HyperbandPruner()

study = optuna.create_study(
    direction="minimize",
    sampler=SelSampler(mode='auto'),
    pruner=pruner,
    #storage="sqlite:///" + f'Optuna/{FileName}_Prdct.db', study_name=f'P{4}',
    load_if_exists=True)
study.optimize(objective, n_trials=1000)
params = list(study.best_params.values())
print('Erro:', study.best_value, 'parameters: ', params)

[I 2026-06-09 10:52:42,226] A new study created in memory with name: no-name-4211c8e8-7e85-4179-876b-da3890dbec02
[I 2026-06-09 10:52:42,243] Trial 0 finished with value: 3.411742558592794e+18 and parameters: {'nI': 42, 'nR': 9, 'nO': 9, 'N1': 0.001, 'N2': 1e-06, 'N3': 0.01, 'τ': 9}. Best is trial 0 with value: 3.411742558592794e+18.
[I 2026-06-09 10:52:42,261] Trial 1 finished with value: 3.4029930857240735e+18 and parameters: {'nI': 23, 'nR': 9, 'nO': 6, 'N1': 0.001, 'N2': 0.01, 'N3': 0.1, 'τ': 23}. Best is trial 1 with value: 3.4029930857240735e+18.
[I 2026-06-09 10:52:42,285] Trial 2 finished with value: 7.523560782404414e+18 and parameters: {'nI': 30, 'nR': 36, 'nO': 10, 'N1': 0.01, 'N2': 0.001, 'N3': 1e-05, 'τ': 25}. Best is trial 1 with value: 3.4029930857240735e+18.
[I 2026-06-09 10:52:42,304] Trial 3 finished with value: 3.160809306571965e+16 and parameters: {'nI': 37, 'nR': 11, 'nO': 7, 'N1': 1e-05, 'N2': 0.01, 'N3': 0.0001, 'τ': 5}. Best is trial 3 with value: 3.160809306571

Erro: 169538102076223.84 parameters:  [6, 2, 10, 1e-06, 0.0001, 0.001, 29]


Erro_M = 0.076862, Erro_R = 0.135507 Parâmetros: {'nI': 38, 'nR': 46, 'nO': 20, 'N1': 0.1, 'N2': 1e-05, 'N3': 0.0001, 'τ': 14, 'mode': 'ahead', 'act': 'tanh'}


In [3]:
params =   {'nI': 38, 'nR': 46, 'nO': 3, 'N1': 0.1, 'N2': 1e-05, 'N3': 0.0001, 'τ': 14, 'mode': 'ahead', 'act': 'tanh'}
params = list(params.values())

In [9]:
nI,nR,nO,N1,N2,N3,τ,mode,act= params
X,Y = PrepareData(sig,nI,nO,mode)
rnn = RTLO(nI,nR,nO,[N1,N2,N3],τ,mode)
rnn.act=act
rnn.ref = len(sig)-nI
for i in range(len(X[:])):
    #rnn.PredRulIntr2(x=X[i],maxRul=len(sig),lim=0.3,store=True,show=False)
    #rnn.PredRul(X[i],maxRul=len(sig),lim=0.75,store=True)

    
    rnn.fit(X[i],Y[i])
print(rnn.εM,rnn.εR)  
PlotPredErrorPLY(rnn,w=1200,h=350)

ValueError: not enough values to unpack (expected 9, got 7)

In [5]:
MAPE(rnn.yR,rnn.yP)

2.910981636431185e+27

In [6]:
rnn.yP.T

array([[ 3.42300832e-02, -6.78765536e-02, -1.29631717e-01, ...,
         1.47624840e+14, -1.81777534e+14,  2.24225935e+14],
       [ 8.28091485e-02,  9.65630089e-02,  4.55225092e+02, ...,
        -1.35605197e+14,  1.66977173e+14, -2.05969416e+14],
       [ 5.06214209e-02,  3.08903583e+02,  7.92438463e+02, ...,
         1.31335737e+14, -1.61719982e+14,  1.99484575e+14]])

In [ ]:
i=int((len(sig)-nI)/2)
r_m = np.mean(rnn.rulR[:i])
r_mL = np.mean(rnn.rulR[:i])*0.35
r_mU = np.mean(rnn.rulR[:i])*1.25
p_m = (np.mean(rnn.rulP[:i]))

print('lower:',r_mL,'mid:',r_m,'upper:',r_mU)
print('pred:',p_m)


εM_m = np.mean(rnn.εM_hist[:i])
εR_M = np.mean(rnn.εR_hist[:i])


print('εM_m:',εM_m)
print('εR_M:',εR_M)

lower: 31.849999999999998 mid: 91.0 upper: 113.75
pred: 27.34426229508197
εM_m: 1.1000916297717287
εR_M: 0.7947284274070086


In [ ]:
#params =  [14, 8, 14, 0.001, 0.01, 1e-06, 1e-07, 29]
nI,nR,nO,N1,N2,N3,τ,mode= params
ηS = [N1,N2,N3]
X,Y = PrepareData(sig,nI,nO,mode)
rnn = RTLO(nI,nR,nO,[N1,N2,N3],τ,mode)
rnn.ref = len(sig)-nI

i=0

In [ ]:
#rnn.PredRul(x=X[i],store=True)
print('iter',i+5)
rnn.PredRulIntr2(x=X[i],maxRul=len(sig),lim=0.75,store=True,show=True)
rnn.fit(X[i],Y[i],store=True,start=0,show=False)
i=i+1